<a href="https://colab.research.google.com/github/tseringTen/funnel-conversion-analysis/blob/main/Untitled13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Closed Deals Table — Cleaning Summary

- Verified key columns unique (0 duplicates) — safe join keys
- Low-null categorical columns filled with "not_specified"
- High-null columns (>90% missing) left as NaN, excluded from analysis
- Won date parsed to datetime, range validated: 2017-12-05 to 2018-11-14
- Behavior profile column: 16 multi-label rows (e.g. "cat, wolf")
  bucketed as "mixed", kept distinct from "not_specified"
- Declared monthly revenue: 797/842 rows (94.7%) had value 0, spread
  across 235 different segment/type combos, no real clustering pattern
  → treated as disguised missing data, converted to NaN, excluded from analysi

In [ ]:
import pandas as pd
import numpy as np
raw_df=pd.read_csv('/content/olist_closed_deals_dataset.csv')

In [ ]:
df=raw_df.copy()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 842 entries, 0 to 841
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   mql_id                         842 non-null    object 
 1   seller_id                      842 non-null    object 
 2   sdr_id                         842 non-null    object 
 3   sr_id                          842 non-null    object 
 4   won_date                       842 non-null    object 
 5   business_segment               841 non-null    object 
 6   lead_type                      836 non-null    object 
 7   lead_behaviour_profile         665 non-null    object 
 8   has_company                    63 non-null     object 
 9   has_gtin                       64 non-null     object 
 10  average_stock                  66 non-null     object 
 11  business_type                  832 non-null    object 
 12  declared_product_catalog_size  69 non-null     flo

In [ ]:
df['mql_id'].duplicated().sum()   # Confirmed both `mql_id` and `seller_id` are unique in
#                                  `closed_deals` (0 duplicates each)


np.int64(0)

In [ ]:
df['seller_id'].duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
mql_id,0
seller_id,0
sdr_id,0
sr_id,0
won_date,0
business_segment,1
lead_type,6
lead_behaviour_profile,177
has_company,779
has_gtin,778


In [ ]:
for col in ['business_segment', 'lead_type', 'business_type', 'lead_behaviour_profile']:    #<25% missing, categorical columns -> fill with placeholder
                                                                                            #mean/mode not valid for text categories; avoids fabricating a specific value)

    df[col] = df[col].fillna('not_specified')                                               #>90% missing (has_company, has_gtin, average_stock, declared_product_catalog_size)
                                                                                             # -> left as NaN, excluded from analysis.

In [ ]:
df.isnull().sum()

,0
mql_id,0
seller_id,0
sdr_id,0
sr_id,0
won_date,0
business_segment,0
lead_type,0
lead_behaviour_profile,0
has_company,779
has_gtin,778


In [ ]:
df['won_date'] = pd.to_datetime(df['won_date'])                   #Parse won_date to datetime and validated range
df['won_date'].min(), df['won_date'].max()

(Timestamp('2017-12-05 02:00:00'), Timestamp('2018-11-14 18:04:19'))

In [ ]:
for col in ['business_segment', 'lead_type', 'business_type', 'lead_behaviour_profile']:
                                                                                 # lead_behaviour_profile contains 16 rows with multi-label values
                                                                                 # (e.g. "cat, wolf") mixed in with single-label values.
                                                                                 # Treating these as their own categories would fragment the data
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

business_segment
business_segment
home_decor                         105
health_beauty                       93
car_accessories                     77
household_utilities                 71
construction_tools_house_garden     69
audio_video_electronics             64
computers                           34
pet                                 30
food_supplement                     28
food_drink                          26
sports_leisure                      25
bed_bath_table                      22
bags_backpacks                      22
toys                                20
fashion_accessories                 19
home_office_furniture               14
phone_mobile                        13
stationery                          13
handcrafted                         12
small_appliances                    12
baby                                10
music_instruments                    9
books                                9
jewerly                              8
watches                       

In [ ]:
multi_label = df['lead_behaviour_profile'].str.contains(',', na=False)  #16 leads were tagged with multiple behavior profiles by the SDR;
                                                                        #grouped as 'mixed' rather than arbitrarily assigning one.
df.loc[multi_label, 'lead_behaviour_profile'] = 'mixed'

In [ ]:
df['lead_behaviour_profile'].value_counts(dropna=False)

,count
lead_behaviour_profile,
cat,407
not_specified,177
eagle,123
wolf,95
shark,24
mixed,16


In [ ]:
(df['declared_monthly_revenue'] == 0).sum()  ## declared_monthly_revenue: 797/842 (94.7%) are exactly 0, spread across

np.int64(797)

In [ ]:
df[df['declared_monthly_revenue'] == 0][['business_segment', 'business_type', 'lead_type']].value_counts()
                                                                      # 235 different segment/type combinations with no real clustering pattern.

business_segment         business_type  lead_type      
car_accessories          reseller       online_medium      30
health_beauty            reseller       online_medium      27
home_decor               manufacturer   online_medium      27
audio_video_electronics  reseller       online_medium      26
household_utilities      reseller       online_medium      18
                                                           ..
stationery               reseller       online_big          1
toys                     reseller       online_beginner     1
                                        online_big          1
watches                  reseller       online_big          1
                                        online_top          1
Name: count, Length: 235, dtype: int64

In [ ]:
df['declared_monthly_revenue'] = df['declared_monthly_revenue'].replace(0, np.nan)  # Treating as disguised missing data rather than genuine self-reported zeros.
df['declared_monthly_revenue'].isnull().sum()

np.int64(797)

In [ ]:
df.to_csv('closed_deal_DATASET.csv',index=False)
from google.colab import files
files.download('closed_deal_DATASET.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>